# Machine Learning - Home Work 3


> Fateme Roshani
810104151

## Question 7 - Feature Conditioning

### Import NumPy and Scikit-learn Dependencies

In [10]:
import numpy as np
from sklearn.feature_selection import VarianceThreshold
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score, StratifiedKFold

### Load Data

### Load Training and Test Features and Labels

In [8]:
train_data = np.loadtxt('trainData.csv', dtype=np.float32, delimiter=',')
train_labels = np.loadtxt('trainLabels.csv', dtype=np.int32, delimiter=',')
test_data = np.loadtxt('testData.csv', dtype=np.float32, delimiter=',')
test_labels = np.loadtxt('testLabels.csv', dtype=np.int32, delimiter=',')
class_names = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']

### Feature Selection

### Filter Low-Variance Features

In [9]:
tr_samples_size, _ = train_data.shape
all_data = np.vstack((train_data,test_data))
sel = VarianceThreshold(threshold=0.90*(1-0.90))
all_data = sel.fit_transform(all_data)
train_data = all_data[:tr_samples_size]
test_data = all_data[tr_samples_size:]

tr_samples_size, feature_size = train_data.shape
te_samples_size, _ = test_data.shape
print('Train Data Samples:',tr_samples_size,
      ', Test Data Samples',te_samples_size,
      ', Feature Size(after feature-selection):', feature_size)

Train Data Samples: 5000 , Test Data Samples 2500 , Feature Size(after feature-selection): 62


### Create Feature Names After Filtering

In [11]:
feature_names = [f"feat_{i}" for i in range(train_data.shape[1])]

### Define Stratified 5-Fold Cross-Validation Accuracy

In [13]:
def evaluate_features(X_subset, y, model=GaussianNB()):
    if X_subset.shape[1] == 0:
        return 0.0
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_subset, y, cv=cv, scoring='accuracy')
    return scores.mean() * 100

### Forward Selection

### Run Greedy Forward Feature Selection

In [ ]:
print("Forward Selection")
selected = []
remaining = list(range(train_data.shape[1]))
forward_steps = [] 

while remaining:
    best_ccr = -1
    best_feat = None
    for feat in remaining:
        current_indices = selected + [feat]
        X_sub = train_data[:, current_indices]
        ccr = evaluate_features(X_sub, train_labels)
        if ccr > best_ccr:
            best_ccr = ccr
            best_feat = feat
    if best_feat is not None:
        selected.append(best_feat)
        remaining.remove(best_feat)
        forward_steps.append((selected.copy(), best_ccr))
        print(f"Step {len(selected)}: Added feature {best_feat}, CCR = {best_ccr:.2f}%")

print("\nForward Selection Results:")
for i, (feats, ccr) in enumerate(forward_steps):
    print(f"Step {i+1}: Features {feats} -> CCR = {ccr:.2f}%")
best_forward_idx = np.argmax([ccr for _, ccr in forward_steps])
best_forward_feats = forward_steps[best_forward_idx][0]
best_forward_ccr = forward_steps[best_forward_idx][1]
print(f"Best forward subset (features indices): {best_forward_feats} with CV CCR = {best_forward_ccr:.2f}%")

Forward Selection
Step 1: Added feature 33, CCR = 23.24%
Step 2: Added feature 34, CCR = 32.84%
Step 3: Added feature 45, CCR = 40.34%
Step 4: Added feature 17, CCR = 47.80%
Step 5: Added feature 26, CCR = 53.68%
Step 6: Added feature 5, CCR = 58.24%
Step 7: Added feature 51, CCR = 62.60%
Step 8: Added feature 14, CCR = 65.68%
Step 9: Added feature 39, CCR = 68.40%
Step 10: Added feature 59, CCR = 70.06%
Step 11: Added feature 28, CCR = 71.44%
Step 12: Added feature 31, CCR = 72.18%
Step 13: Added feature 1, CCR = 73.62%
Step 14: Added feature 12, CCR = 74.90%
Step 15: Added feature 37, CCR = 76.10%
Step 16: Added feature 20, CCR = 77.08%
Step 17: Added feature 8, CCR = 78.04%
Step 18: Added feature 55, CCR = 78.68%
Step 19: Added feature 25, CCR = 79.20%
Step 20: Added feature 19, CCR = 79.48%
Step 21: Added feature 49, CCR = 80.06%
Step 22: Added feature 16, CCR = 80.50%
Step 23: Added feature 41, CCR = 81.00%
Step 24: Added feature 24, CCR = 81.24%
Step 25: Added feature 53, CCR = 8

### Backward Elimination

### Run Greedy Backward Feature Elimination

In [ ]:
print("\nBackward Elimination")
current = list(range(train_data.shape[1]))
backward_steps = []

full_ccr = evaluate_features(train_data[:, current], train_labels)
backward_steps.append((current.copy(), full_ccr))
print(f"Step 0 (all features): CCR = {full_ccr:.2f}%")

while len(current) > 1:
    best_ccr = -1
    worst_feat = None
    for feat in current:
        temp = [f for f in current if f != feat]
        X_sub = train_data[:, temp]
        ccr = evaluate_features(X_sub, train_labels)
        if ccr > best_ccr:
            best_ccr = ccr
            worst_feat = feat
    if worst_feat is not None:
        current.remove(worst_feat)
        backward_steps.append((current.copy(), best_ccr))
        print(f"Step {len(backward_steps)-1}: Removed feature {worst_feat}, CCR = {best_ccr:.2f}%")

print("\nBackward Elimination Results:")
for i, (feats, ccr) in enumerate(backward_steps):
    print(f"Step {i}: Features {feats} -> CCR = {ccr:.2f}%")
best_backward_idx = np.argmax([ccr for _, ccr in backward_steps])
best_backward_feats = backward_steps[best_backward_idx][0]
best_backward_ccr = backward_steps[best_backward_idx][1]
print(f"Best backward subset (features indices): {best_backward_feats} with CV CCR = {best_backward_ccr:.2f}%")


Backward Elimination
Step 0 (all features): CCR = 80.84%
Step 1: Removed feature 43, CCR = 81.54%
Step 2: Removed feature 22, CCR = 81.84%
Step 3: Removed feature 52, CCR = 82.04%
Step 4: Removed feature 58, CCR = 82.30%
Step 5: Removed feature 4, CCR = 82.32%
Step 6: Removed feature 49, CCR = 82.50%
Step 7: Removed feature 9, CCR = 82.64%
Step 8: Removed feature 61, CCR = 82.72%
Step 9: Removed feature 42, CCR = 82.68%
Step 10: Removed feature 1, CCR = 82.80%
Step 11: Removed feature 57, CCR = 83.04%
Step 12: Removed feature 36, CCR = 83.08%
Step 13: Removed feature 44, CCR = 83.24%
Step 14: Removed feature 31, CCR = 83.32%
Step 15: Removed feature 7, CCR = 83.38%
Step 16: Removed feature 54, CCR = 83.38%
Step 17: Removed feature 40, CCR = 83.32%
Step 18: Removed feature 21, CCR = 83.30%
Step 19: Removed feature 59, CCR = 83.28%
Step 20: Removed feature 56, CCR = 83.28%
Step 21: Removed feature 28, CCR = 83.14%
Step 22: Removed feature 23, CCR = 83.28%
Step 23: Removed feature 11, CC

### Evaluate the best subset on the test set

### Evaluate the Forward-Selected Features on the Test Set

In [18]:
best_subset = best_forward_feats 
model = GaussianNB()
model.fit(train_data[:, best_subset], train_labels)
test_pred = model.predict(test_data[:, best_subset])
test_ccr = np.mean(test_pred == test_labels) * 100
print(f"\nFinal test CCR using best subset (indices {best_subset}): {test_ccr:.2f}%")


Final test CCR using best subset (indices [33, 34, 45, 17, 26, 5, 51, 14, 39, 59, 28, 31, 1, 12, 37, 20, 8, 55, 25, 19, 49, 16, 41, 24, 53, 13, 3, 10, 42, 54, 32, 48, 58, 40]): 77.40%
